In [ ]:
# prompt: bidirectionnal LSTM tutorial with pytorch

import torch
import torch.nn as nn
import torch.optim as optim

# Define the parameters
input_size = 10  # Size of input features
hidden_size = 20  # Size of hidden state
num_layers = 2  # Number of LSTM layers
output_size = 1  # Size of output

# Define the BiLSTM model
class BiLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(BiLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        # The batch_first=True argument makes the input/output tensors have shape (batch, sequence, features)
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True)
        # The output of a bidirectional LSTM has a size of 2 * hidden_size
        self.fc = nn.Linear(hidden_size * 2, output_size)

    def forward(self, x):
        # Set initial hidden and cell states
        # The hidden and cell states for a bidirectional LSTM have shape (2 * num_layers, batch, hidden_size)
        h0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size).to(x.device)

        # Forward propagate LSTM
        # out: tensor of shape (batch_size, seq_length, hidden_size*2)
        out, _ = self.lstm(x, (h0, c0))

        # Decode the hidden state of the last time step
        # We take the output from the last time step (out[:, -1, :])
        out = self.fc(out[:, -1, :])
        return out

# Create a dummy dataset (replace with your actual data)
# Input tensor with shape (batch_size, seq_length, input_size)
batch_size = 4
seq_length = 5
input_data = torch.randn(batch_size, seq_length, input_size)
target_data = torch.randn(batch_size, output_size)

# Instantiate the model
model = BiLSTM(input_size, hidden_size, num_layers, output_size)

# Define loss function and optimizer
criterion = nn.MSELoss()  # Mean Squared Error Loss for regression
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop (example)
num_epochs = 100
for epoch in range(num_epochs):
    # Forward pass
    outputs = model(input_data)
    loss = criterion(outputs, target_data)

    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

# Example prediction
with torch.no_grad():
    test_input = torch.randn(1, seq_length, input_size)
    prediction = model(test_input)
    print(f'\nExample prediction: {prediction.item():.4f}')
